In [2]:
# Data Manipulation & Math 
import math  
import pandas as pd  

# System & Performance Benchmarking 
import os  
import psutil  
import time  

# Statistical Analysis 
from scipy.stats import chisquare  

#MT19937
import random

Implementation of MT19937
- Uses Python's built-in MT19937 (Mersenne Twister) pseudorandom number generator. It is initialized with a seed, and each call to extract_number() returns a 32-bit pseudorandom integer.

In [3]:
import random

class MT19937:

    def __init__(self, seed):

        self.rng = random.Random(seed)

    def extract_number(self):

        return self.rng.getrandbits(32)

Generates a pseudorandom bit sequence of the requested length by repeatedly extracting 32-bit integers from the MT19937 generator, converting each integer to its 32-bit binary representation, and concatenating the bits until the desired number of bits is obtained.

In [4]:
def generate_bits(mt, num_bits):
    bits = []

    while len(bits) < num_bits:
        value = mt.extract_number()

        bits.extend(f"{value:032b}")

    return "".join(bits[:num_bits])

Benchmarks the MT19937 pseudorandom number generator by generating bitstreams of different lengths over multiple trials while measuring execution time, CPU time, throughput, and memory usage. 

The generated bitstream and performance metrics for each trial are stored and returned for further analysis.

In [5]:
def benchmark_mt19937(bit_sizes, trials=10, seed=5489):

    process = psutil.Process(os.getpid())
    results = []

    for bits in bit_sizes:

        print(f"Benchmarking {bits} bits...")

        for trial in range(1, trials + 1):

            mt = MT19937(seed + trial)

            cpu_start = time.process_time()
            wall_start = time.perf_counter()

            bitstream = generate_bits(mt, bits)

            wall_end = time.perf_counter()
            cpu_end = time.process_time()

            runtime = wall_end - wall_start
            cpu_time = cpu_end - cpu_start
            throughput = bits / runtime

            peak_ram = process.memory_info().rss / (1024 * 1024)

            results.append({
                "Bits": bits,
                "Trial": trial,
                "Runtime (s)": runtime,
                "CPU Time (s)": cpu_time,
                "Throughput (bits/s)": throughput,
                "Peak RAM (MB)": peak_ram,
                "Bitstream": bitstream
            })

    return results

Defines the bitstream sizes to be tested and executes the MT19937 benchmarking function for each size over 10 trials. 

The resulting performance metrics and generated bitstreams are stored in the results variable for subsequent analysis.

In [6]:
bit_sizes = [10000, 20000, 40000, 80000]

results = benchmark_mt19937(
    bit_sizes=bit_sizes,
    trials=10
)

Benchmarking 10000 bits...
Benchmarking 20000 bits...
Benchmarking 40000 bits...
Benchmarking 80000 bits...


The below three cells evaluate the statistical quality of the generated pseudorandom bitstreams. They compute the Shannon entropy and min-entropy to quantify the randomness and unpredictability of the sequences, and perform a chi-squared goodness-of-fit test to compare the observed distribution of 0s and 1s against the expected uniform distribution. 

Together, these metrics provide an assessment of the randomness and statistical properties of the generated bitstreams.

In [7]:
def shannon_entropy(bitstream):

    total = len(bitstream)

    count0 = bitstream.count('0')
    count1 = bitstream.count('1')

    entropy = 0

    for count in [count0, count1]:

        if count == 0:
            continue

        p = count / total
        entropy -= p * math.log2(p)

    return entropy

In [8]:
def min_entropy(bitstream):

    total = len(bitstream)

    count0 = bitstream.count('0')
    count1 = bitstream.count('1')

    pmax = max(count0, count1) / total

    return -math.log2(pmax)

In [9]:
def chi_squared_test(bitstream):

    observed = [
        bitstream.count('0'),
        bitstream.count('1')
    ]

    expected = [
        len(bitstream) / 2,
        len(bitstream) / 2
    ]

    statistic, p_value = chisquare(
        observed,
        expected
    )

    return statistic, p_value

The below cell computes the statistical quality metrics for each generated bitstream and combines them with the recorded performance metrics into a single DataFrame. 

The resulting table contains
- runtime
- CPU time
- throughput
- memory usage
- Shannon entropy
- min-entropy
- chi-squared statistic
- p-value

for every trial.

In [10]:
df = pd.DataFrame(results)

rows = []

for result in results:

    entropy = shannon_entropy(result["Bitstream"])
    min_ent = min_entropy(result["Bitstream"])
    chi2, p_value = chi_squared_test(result["Bitstream"])

    rows.append({
        "Bits": result["Bits"],
        "Trial": result["Trial"],
        "Runtime (s)": result["Runtime (s)"],
        "CPU Time (s)": result["CPU Time (s)"],
        "Throughput (bits/s)": result["Throughput (bits/s)"],
        "Peak RAM (MB)": result["Peak RAM (MB)"],
        "Shannon Entropy": entropy,
        "Min Entropy": min_ent,
        "Chi-Squared": chi2,
        "p-value": p_value
    })

df = pd.DataFrame(rows)

display(df)

,Bits,Trial,Runtime (s),CPU Time (s),Throughput (bits/s),Peak RAM (MB),Shannon Entropy,Min Entropy,Chi-Squared,p-value
0,10000,1,0.000458,0.000000,2.184360e+07,173.367188,0.999977,0.991943,0.31360,0.575479
1,10000,2,0.000323,0.000000,3.094059e+07,173.367188,0.999717,0.971714,3.92040,0.047704
2,10000,3,0.001805,0.000000,5.539246e+06,173.367188,0.999999,0.997982,0.01960,0.888660
3,10000,4,0.000507,0.000000,1.973165e+07,173.367188,0.999944,0.987360,0.77440,0.378859
4,10000,5,0.000465,0.000000,2.149613e+07,173.367188,0.999512,0.962969,6.76000,0.009322
5,10000,6,0.000293,0.000000,3.412969e+07,173.367188,0.999967,0.990223,0.46240,0.496504
6,10000,7,0.000305,0.000000,3.273322e+07,173.367188,1.000000,1.000000,0.00000,1.000000
7,10000,8,0.000286,0.000000,3.490402e+07,173.367188,1.000000,0.998846,0.00640,0.936237
8,10000,9,0.000286,0.000000,3.497727e+07,173.367188,0.999922,0.985073,1.08160,0.298340
9,10000,10,0.000409,0.000000,2.445586e+07,173.371094,0.999982,0.992804,0.25000,0.617075


Saves the detailed benchmark results to a CSV file and generates a summary table by bitstream size. It computes the mean and standard deviation of the performance and randomness metrics across all trials for easy comparison.

In [11]:
df.to_csv("mt19937_results.csv", index=False)

summary_df = (
    df.groupby("Bits")
      .agg({
          "Runtime (s)": ["mean", "std"],
          "CPU Time (s)": ["mean", "std"],
          "Throughput (bits/s)": ["mean", "std"],
          "Peak RAM (MB)": ["mean", "std"],
          "Shannon Entropy": ["mean", "std"],
          "Min Entropy": ["mean", "std"],
          "Chi-Squared": ["mean"],
          "p-value": ["mean"]
      })
)

summary_df

Runtime (s)           CPU Time (s)           Throughput (bits/s)  \
             mean       std         mean       std                mean   
Bits                                                                     
10000    0.000514  0.000461     0.000000  0.000000        2.607513e+07   
20000    0.001143  0.000329     0.001563  0.004941        1.880331e+07   
40000    0.002049  0.000649     0.003125  0.006588        2.169613e+07   
80000    0.005003  0.000789     0.004687  0.007548        1.636301e+07   

                    Peak RAM (MB)           Shannon Entropy            \
                std          mean       std            mean       std   
Bits                                                                    
10000  9.389910e+06    173.367578  0.001235        0.999902  0.000161   
20000  5.370611e+06    173.394531  0.000000        0.999976  0.000044   
40000  7.733240e+06    173.641406  0.105602        0.999994  0.000008   
80000  2.632857e+06    175.545312  0.169723        0.999995  0.000006   

      Min Entropy           Chi-Squared   p-value  
             mean       std        mean      mean  
Bits                                               
10000    0.987892  0.012044     1.35884  0.524818  
20000    0.994008  0.006134     0.67758  0.620223  
40000    0.996609  0.002723     0.35079  0.660488  
80000    0.996968  0.002239     0.52871  0.586835

Saves the generated bitstreams as individual text files and exports the aggregated benchmark summary to a JSON file for future analysis and visualization.

In [12]:
# Create a directory to store all generated bitstreams
output_dir = "bitstreams"
os.makedirs(output_dir, exist_ok=True)

for result in results:
    bits = result["Bits"]
    trial = result["Trial"]

    filename = os.path.join(
        output_dir,
        f"mt19937_{bits}_trial{trial}.txt"
    )

    with open(filename, "w") as f:
        f.write(result["Bitstream"])

#preparing aggregated benchmark data, saving to a JSON file
json_df = summary_df.copy()
json_df.columns = [f"{col[0]}_{col[1]}" for col in json_df.columns]
json_df = json_df.reset_index()

json_df.to_json("mt19937_summary.json", orient="records", indent=4)